# Review frog & toad detections by site

1. **Pick sites** on a map.
2. **Sample** 20 frog/toad detections (confidence ≥ 0.5) at each selected site.
3. **Review** the clips in `jupyter_bioacoustic`.

Uses the same API token (`API_TOKEN` in `.env`) and audio helpers ([`api_audio.py`](api_audio.py)) as `01_review_detections.ipynb`. The map works in VS Code or JupyterLab; the annotator only works in JupyterLab.

## 1. Pick sites

Recordings only have a `deployment_id`. The site name (`placename`) and coordinates are on the deployment, so we join the two and plot one point per site.

**Selecting:** click a site, shift+click to add more, or use the lasso/box tools in the map's toolbar (hold shift to add to the selection). Selected sites turn red.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import api_audio
from api_audio import query_api

API_URL = "https://api.wildlifesoundhub.org"
api_audio.connect(API_URL, token_env="API_TOKEN", cookie_name="__Secure-authjs.session-token")

deployments = pd.DataFrame(query_api("/deployments/")).rename(columns={"id": "deployment_id", "placename": "site"})
deployments[["lon", "lat"]] = [g["coordinates"] for g in deployments.geometry]
recordings = pd.DataFrame(query_api("/recordings/")).merge(
    deployments[["deployment_id", "site", "lat", "lon"]], on="deployment_id"
)
sites = recordings.groupby("site", as_index=False).agg(
    lat=("lat", "mean"), lon=("lon", "mean"), recordings=("id", "size")
)


# Panning/zooming the map sends plotly.js's internal "map._derived" key, which plotly.py
# rejects with "Invalid property path" on every move. Drop it.
class SiteMap(go.FigureWidget):
    def plotly_relayout(self, relayout_data, **kwargs):
        relayout_data = {k: v for k, v in relayout_data.items() if not k.endswith("._derived")}
        super().plotly_relayout(relayout_data, **kwargs)


fig = SiteMap(
    px.scatter_map(
        sites, lat="lat", lon="lon", hover_name="site", hover_data=["recordings"],
        center=dict(lat=sites.lat.mean(), lon=sites.lon.mean()), zoom=5, height=600,
    )
)
fig.update_traces(marker_size=10, selected_marker_color="crimson")
fig.update_layout(clickmode="event+select", margin=dict(l=0, r=0, t=0, b=0))
fig

## 2. Sample frog & toad detections

The detections API needs a deployment and only matches `common_name` exactly. So for each deployment at the selected sites we list its species, keep the frog/toad names, and fetch those detections. Then we take a random 20 per site.

Rerun this cell after changing the selection on the map.

In [2]:
import re
from concurrent.futures import ThreadPoolExecutor

SPECIES_PATTERN = r"(frog|toad)\b"  # word end: keeps "Squirrel Treefrog", skips the bird "Tawny Frogmouth"
MIN_CONFIDENCE = 0.5
CLIPS_PER_SITE = 5

selected_sites = sites.site.iloc[list(fig.data[0].selectedpoints or [])].tolist()
assert selected_sites, "Select at least one site on the map above"
print(f"{len(selected_sites)} sites:", ", ".join(selected_sites))


def frog_toad_detections(deployment_id):
    """All frog/toad detections >= MIN_CONFIDENCE for one deployment."""
    species = query_api("/detections/", deployment=deployment_id, confidence=MIN_CONFIDENCE, count=True)
    names = [s["common_name"] for s in species if re.search(SPECIES_PATTERN, s["common_name"], re.IGNORECASE)]
    rows = [
        det
        for name in names
        for det in query_api("/detections/", deployment=deployment_id, confidence=MIN_CONFIDENCE, common_name=name)
    ]
    return pd.DataFrame(rows).assign(deployment_id=deployment_id)


deployment_site = recordings[recordings.site.isin(selected_sites)].groupby("deployment_id").site.first()
with ThreadPoolExecutor(max_workers=8) as pool:  # each API call takes 5-20 s, so run them in parallel
    hits = pd.concat(pool.map(frog_toad_detections, deployment_site.index), ignore_index=True)
assert len(hits), "No frog/toad detections at the selected sites"
hits["site"] = hits.deployment_id.map(deployment_site)

clips = (
    hits.sample(frac=1, random_state=0)  # shuffle, so head() takes a random sample per site
    .groupby("site")
    .head(CLIPS_PER_SITE)
    .sort_values(["site", "confidence"], ascending=[True, False])
    .reset_index(drop=True)
)
clips["id"] = clips.recording_id.astype(str) + "_" + clips.start_time.astype(str) + "_" + clips.common_name
# The widget reads audio from this column; install_widget_audio() below resolves it to S3.
clips["audio_url"] = API_URL + "/recordings/" + clips.recording_id.astype(str)

print(f"{len(hits):,} frog/toad detections at these sites, sampled {len(clips)}")
clips.groupby(["site", "common_name"]).size().rename("clips").to_frame()

1 sites: FCWA1
4 frog/toad detections at these sites, sampled 4


clips
site  common_name                 
FCWA1 Green Frog                 1
      Northern Cricket Frog      2
      Pine Woods Treefrog        1

## 3. Review

Reviews are saved to `../data/frog_toad_reviews.csv`.

In [3]:
from jupyter_bioacoustic import BioacousticAnnotator

api_audio.install_widget_audio()  # lets the widget read the API recording URLs in audio_url

ann = BioacousticAnnotator(
    data=clips,
    data_index_column="id",
    audio_column="audio_url",
    display_columns=["site", "common_name", "confidence", "recording_id", "start_time", "end_time"],
    info_card_title="[[common_name]]",
    default_buffer=2,  # seconds of padding on each side of the detection
    output="../data/frog_toad_reviews.csv",
)
ann.open()

<IPython.core.display.Javascript object>